In [1]:
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

C:\Users\nosen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
!pip uninstall sentence-transformers transformers protobuf -y
!pip install sentence-transformers transformers protobuf==3.20.*

In [2]:
class PlayerEmbeddingFunction(EmbeddingFunction):
    def __init__(self, model_path: str):
        # Inizializza il modello fine-tunato
        self.model = SentenceTransformer(model_path)

    def __call__(self, input: Documents) -> Embeddings:
        # Calcola gli embeddings usando il modello
        embeddings = self.model.encode(input).tolist()  # Converte gli embeddings in formato lista
        return embeddings

# Crea un'istanza della classe con il percorso al modello
embedding_function = PlayerEmbeddingFunction(model_path="Model")


In [10]:
!pip install --upgrade chroma-migrate
!python -m chroma_migrate

  Using cached chroma_migrate-0.0.7-py3-none-any.whl (12 kB)
  Using cached clickhouse_connect-0.6.6-cp310-cp310-win_amd64.whl (227 kB)
  Using cached chroma_bullet-2.2.0-py3-none-any.whl (11 kB)
  Using cached more_itertools-10.5.0-py3-none-any.whl (60 kB)
  Using cached lz4-4.3.3-cp310-cp310-win_amd64.whl (99 kB)
  Using cached tokenizers-0.20.3-cp310-none-win_amd64.whl (2.4 MB)
  Using cached protobuf-5.29.2-cp310-abi3-win_amd64.whl (434 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3


ERROR: Could not install packages due to an OSError: [WinError 5] Accesso negato: 'C:\\Users\\nosen\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python310\\site-packages\\google\\~.otobuf\\internal\\_api_implementation.cp310-win_amd64.pyd'
Check the permissions.


[notice] A new release of pip is available: 23.1.2 -> 24.3.1
[notice] To update, run: C:\Users\nosen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
C:\Users\nosen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe: No module named chroma_migrate


In [13]:
embedding_function = PlayerEmbeddingFunction(model_path="Model")

client = chromadb.PersistentClient(path="/VectorDB")

# Crea una collection usando la funzione di embedding personalizzata
collection = client.get_or_create_collection(
    name="player_embeddings",
    embedding_function=embedding_function
)

In [29]:
texts = ["Esempio di testo 1", "difensore", "pass", 'retro']

collection.upsert(
    documents=texts,
    ids=["id1", "id2", "id3", "id4"]
)

# Interroga la collection
query = "passaggio"
results = collection.query(
    query_texts=[query],
    n_results=3  # Numero di risultati desiderati
)

# Stampa i risultati
for doc, score in zip(results["documents"], results["distances"]):
    print(f"Documento: {doc}, Similarità: {score}")

Documento: ['pass', 'Esempio di testo 1', 'difensore'], Similarità: [64.08396364522183, 235.07347735004842, 280.96526759856755]
